<a href="https://www.kaggle.com/code/mradulnamdeo/iit-delhi-translation-task?scriptVersionId=322089449" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
%%writefile /kaggle/working/architecture.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def _norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        output = self._norm(x.float()).type_as(x)
        return output * self.weight

class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len=2048, base=10000.0):
        super().__init__()
        self.head_dim = head_dim
        self.max_seq_len = max_seq_len
        self.base = base
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq)
        self.build_cache(max_seq_len)

    def build_cache(self, seq_len):
        t = torch.arange(seq_len, device=self.inv_freq.device, dtype=self.inv_freq.dtype)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos()[None, None, :, :])
        self.register_buffer("sin_cached", emb.sin()[None, None, :, :])

    def forward(self, q, k, seq_len):
        if seq_len > self.max_seq_len:
            self.build_cache(seq_len)
        cos = self.cos_cached[:, :, :seq_len, ...]
        sin = self.sin_cached[:, :, :seq_len, ...]
        def rotate_half(x):
            x1 = x[..., : x.shape[-1] // 2]
            x2 = x[..., x.shape[-1] // 2 :]
            return torch.cat((-x2, x1), dim=-1)
        q_embed = (q * cos) + (rotate_half(q) * sin)
        k_embed = (k * cos) + (rotate_half(k) * sin)
        return q_embed, k_embed

class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len=2048):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.head_dim = d_model // n_heads
        self.wq = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(n_heads * self.head_dim, d_model, bias=False)
        self.rope = RotaryPositionalEmbedding(self.head_dim, max_seq_len)

    def forward(self, x, mask=None):
        batch_size, seq_len, d_model = x.shape
        xq = self.wq(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        xk = self.wk(x).view(batch_size, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)
        xv = self.wv(x).view(batch_size, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)
        xq, xk = self.rope(xq, xk, seq_len)
        xk = xk.repeat_interleave(self.n_rep, dim=1)
        xv = xv.repeat_interleave(self.n_rep, dim=1)
        scores = torch.matmul(xq, xk.transpose(2, 3)) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores + mask
        scores = F.softmax(scores.float(), dim=-1).type_as(xq)
        output = torch.matmul(scores, xv)
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        return self.wo(output)

class FeedForward(nn.Module):
    def __init__(self, d_model, hidden_dim, dropout=0.1):
        super().__init__()
        self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, d_model, bias=False)
        self.w3 = nn.Linear(d_model, hidden_dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w2(F.silu(self.w1(x)) * self.w3(x)))

Writing /kaggle/working/architecture.py


In [2]:
%%writefile pretrain_encoder.py
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
import os
import math
from tqdm import tqdm

from architecture import RMSNorm, RotaryPositionalEmbedding, GroupedQueryAttention, FeedForward

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

class CustomEncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, hidden_dim, max_seq_len):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attention = GroupedQueryAttention(d_model, n_heads, n_kv_heads, max_seq_len)
        self.norm2 = RMSNorm(d_model)
        self.feed_forward = FeedForward(d_model, hidden_dim)

    def forward(self, x, mask=None):
        h = x + self.attention(self.norm1(x), mask)
        out = h + self.feed_forward(self.norm2(h))
        return out

class CustomBERTEncoder(nn.Module):
    def __init__(self, vocab_size, d_model=768, n_layers=12, n_heads=12, n_kv_heads=4, hidden_dim=3072, max_seq_len=256):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([CustomEncoderBlock(d_model, n_heads, n_kv_heads, hidden_dim, max_seq_len) for _ in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.mlm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.mlm_head.weight = self.token_embedding.weight

    def forward(self, input_ids, mask=None):
        x = self.token_embedding(input_ids)
        for layer in self.layers:
            x = layer(x, mask)
        x = self.final_norm(x)
        logits = self.mlm_head(x)
        return logits, x 

class MLMDataset(Dataset):
    def __init__(self, file_paths, tokenizer, max_len=128, mask_prob=0.15):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.mask_prob = mask_prob
        self.lines = []
        print("Reading data for Encoder Pretraining...")
        for path in file_paths:
            if os.path.exists(path):
                with open(path, 'r', encoding='utf-8') as f:
                    self.lines.extend([line.strip() for line in f.readlines() if len(line.strip()) > 0])
            else:
                print(f"[Warning] Could not find {path}")
        print(f"Loaded {len(self.lines)} total lines for MLM.")

    def __len__(self): return len(self.lines)

    def __getitem__(self, idx):
        text = str(self.lines[idx])
        encoding = self.tokenizer(text, truncation=True, max_length=self.max_len, padding='max_length', return_tensors="pt")
        input_ids = encoding['input_ids'].squeeze(0)
        labels = input_ids.clone()
        probability_matrix = torch.full(labels.shape, self.mask_prob)
        special_tokens_mask = [1 if id in self.tokenizer.all_special_ids else 0 for id in labels.tolist()]
        probability_matrix.masked_fill_(torch.tensor(special_tokens_mask, dtype=torch.bool), value=0.0)
        masked_indices = torch.bernoulli(probability_matrix).bool()
        labels[~masked_indices] = -100  
        indices_replaced = torch.bernoulli(torch.full(labels.shape, 0.8)).bool() & masked_indices
        input_ids[indices_replaced] = self.tokenizer.mask_token_id
        indices_random = torch.bernoulli(torch.full(labels.shape, 0.5)).bool() & masked_indices & ~indices_replaced
        random_words = torch.randint(len(self.tokenizer), labels.shape, dtype=torch.long)
        input_ids[indices_random] = random_words[indices_random]
        return {'input_ids': input_ids, 'labels': labels}

def pretrain_encoder():
    print("Initializing Hindi Tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained("l3cube-pune/hindi-bert-v2")
    
    files = [
        "/kaggle/input/datasets/mradulnamdeo/iit-delhi-adivaani/train.hi", 
        "/kaggle/input/datasets/mradulnamdeo/iit-delhi-adivaani/test.hi"
    ]
    
    dataset = MLMDataset(files, tokenizer, max_len=128)
    # REDUCED BATCH SIZE TO PREVENT OOM
    dataloader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=2)
    
    model = CustomBERTEncoder(vocab_size=tokenizer.vocab_size).to(device)
    param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nEncoder Model Initialized. Parameters: {param_count:,}")
    
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss(ignore_index=-100)
    scaler = torch.amp.GradScaler('cuda') 

    epochs = 4
    os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
    print("\nStarting Encoder MLM Pretraining...")
    model.train()
    
    for epoch in range(epochs):
        total_loss = 0
        loop = tqdm(dataloader, leave=True)
        for batch in loop:
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                logits, _ = model(input_ids)
                loss = criterion(logits.view(-1, tokenizer.vocab_size), labels.view(-1))
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
            loop.set_description(f"Epoch [{epoch+1}/{epochs}]")
            loop.set_postfix(loss=loss.item())
        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1} Completed | Average MLM Loss: {avg_loss:.4f}")
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'loss': avg_loss}, f"/kaggle/working/checkpoints/encoder_110M_epoch_{epoch+1}.pt")
    print("\nEncoder Pretraining Complete.")

if __name__ == "__main__":
    pretrain_encoder()

Writing pretrain_encoder.py


In [3]:
%%writefile pretrain_decoder.py
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
import os
from tqdm import tqdm

from architecture import RMSNorm, RotaryPositionalEmbedding, GroupedQueryAttention, FeedForward

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

class CustomDecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, hidden_dim, max_seq_len):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.self_attention = GroupedQueryAttention(d_model, n_heads, n_kv_heads, max_seq_len)
        self.norm2 = RMSNorm(d_model)
        self.feed_forward = FeedForward(d_model, hidden_dim)

    def forward(self, x, causal_mask):
        h = x + self.self_attention(self.norm1(x), mask=causal_mask)
        out = h + self.feed_forward(self.norm2(h))
        return out

class CustomGPTDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=768, n_layers=12, n_heads=12, n_kv_heads=4, hidden_dim=3072, max_seq_len=256):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([CustomDecoderBlock(d_model, n_heads, n_kv_heads, hidden_dim, max_seq_len) for _ in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape
        causal_mask = torch.triu(torch.full((seq_len, seq_len), float('-inf'), device=input_ids.device), diagonal=1).unsqueeze(0).unsqueeze(0)
        x = self.token_embedding(input_ids)
        for layer in self.layers:
            x = layer(x, causal_mask)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        return logits, x

class CLMDataset(Dataset):
    def __init__(self, file_paths, tokenizer, max_len=128):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.lines = []
        print("Reading data for Decoder Pretraining (Marathi only)...")
        for path in file_paths:
            if os.path.exists(path):
                with open(path, 'r', encoding='utf-8') as f:
                    self.lines.extend([line.strip() for line in f.readlines() if len(line.strip()) > 0])
            else:
                print(f"[Warning] Could not find {path}")
        print(f"Loaded {len(self.lines)} total lines for CLM.")

    def __len__(self): return len(self.lines)

    def __getitem__(self, idx):
        text = str(self.lines[idx])
        encoding = self.tokenizer(text, truncation=True, max_length=self.max_len, padding='max_length', return_tensors="pt")
        input_ids = encoding['input_ids'].squeeze(0)
        return {'input_ids': input_ids}

def pretrain_decoder():
    print("Initializing Marathi Tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained("l3cube-pune/marathi-bert-v2")
    
    files = [
        "/kaggle/input/datasets/mradulnamdeo/iit-delhi-adivaani/train.mr", 
        "/kaggle/input/datasets/mradulnamdeo/iit-delhi-adivaani/test.mr"
    ]
    
    dataset = CLMDataset(files, tokenizer, max_len=128)
    # REDUCED BATCH SIZE TO PREVENT OOM
    dataloader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=2)
    
    model = CustomGPTDecoder(vocab_size=tokenizer.vocab_size).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
    scaler = torch.amp.GradScaler('cuda') 

    epochs = 4
    os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
    print("\nStarting Decoder Autoregressive Pretraining...")
    model.train()
    
    for epoch in range(epochs):
        total_loss = 0
        loop = tqdm(dataloader, leave=True)
        for batch in loop:
            input_ids = batch['input_ids'].to(device)
            x = input_ids[:, :-1].contiguous()
            y = input_ids[:, 1:].contiguous()
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                logits, _ = model(x)
                loss = criterion(logits.view(-1, tokenizer.vocab_size), y.view(-1))
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
            loop.set_description(f"Epoch [{epoch+1}/{epochs}]")
            loop.set_postfix(loss=loss.item())
        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1} Completed | Average CLM Loss: {avg_loss:.4f}")
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'loss': avg_loss}, f"/kaggle/working/checkpoints/decoder_124M_epoch_{epoch+1}.pt")
    print("\nDecoder Pretraining Complete.")

if __name__ == "__main__":
    pretrain_decoder()

Writing pretrain_decoder.py


In [4]:
%cd /kaggle/working
!python pretrain_encoder.py
!python pretrain_decoder.py
!python train_translation.py

/kaggle/working


In [5]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename in ['train.hi', 'train.mr', 'test.hi', 'test.mr']:
            print(os.path.join(dirname, filename))

/kaggle/input/datasets/mradulnamdeo/iit-delhi-adivaani/train.hi
/kaggle/input/datasets/mradulnamdeo/iit-delhi-adivaani/train.mr
/kaggle/input/datasets/mradulnamdeo/iit-delhi-adivaani/test.mr
/kaggle/input/datasets/mradulnamdeo/iit-delhi-adivaani/test.hi


In [6]:
import os
print("Finding your checkpoints...\n")
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename.endswith('.pt'):
            print(os.path.join(dirname, filename))

Finding your checkpoints...

/kaggle/input/datasets/akshy10/encoder-run/encoder_110M_epoch_4.pt
/kaggle/input/datasets/akshy10/decpder-run/decoder_124M_epoch_4.pt
/kaggle/input/datasets/mradulnamdeo/translation-run-data/final_translation_model_epoch_2.pt
/kaggle/input/datasets/mradulnamdeo/translation-run-data/final_translation_model_epoch_1.pt
/kaggle/input/datasets/mradulnamdeo/translation-run-data/final_translation_model_epoch_3.pt


In [7]:
%%writefile train_translation.py
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
import os
from tqdm import tqdm

from architecture import RMSNorm, RotaryPositionalEmbedding, GroupedQueryAttention, FeedForward
from pretrain_encoder import CustomBERTEncoder
from pretrain_decoder import CustomGPTDecoder

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

class ConnectedTranslationModel(nn.Module):
    def __init__(self, encoder, decoder, d_model=768, n_heads=12):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        
        # True Query-Key-Value Cross Attention
        self.cross_attention_layers = nn.ModuleList([
            nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, batch_first=True) 
            for _ in range(len(decoder.layers))
        ])
        self.cross_norms = nn.ModuleList([RMSNorm(d_model) for _ in range(len(decoder.layers))])

    def forward(self, src_ids, tgt_ids, src_mask=None, tgt_mask=None):
        batch_size, seq_len_tgt = tgt_ids.shape
        
        extended_src_mask = (1.0 - src_mask.unsqueeze(1).unsqueeze(2)) * -1e9 if src_mask is not None else None
        if extended_src_mask is not None: extended_src_mask = extended_src_mask.to(device)

        _, enc_states = self.encoder(src_ids, mask=extended_src_mask) 
        
        cross_pad_mask = (src_mask == 0).to(device) if src_mask is not None else None
        causal_mask = torch.triu(torch.full((seq_len_tgt, seq_len_tgt), float('-inf'), device=device), diagonal=1).unsqueeze(0).unsqueeze(0)
        
        dec_x = self.decoder.token_embedding(tgt_ids)
        
        for i, layer in enumerate(self.decoder.layers):
            h = dec_x + layer.self_attention(layer.norm1(dec_x), mask=causal_mask)
            
            normed_h = self.cross_norms[i](h)
            attn_output, _ = self.cross_attention_layers[i](
                query=normed_h, 
                key=enc_states, 
                value=enc_states, 
                key_padding_mask=cross_pad_mask,
                need_weights=False
            )
            c = h + attn_output
            dec_x = c + layer.feed_forward(layer.norm2(c))
            
        dec_x = self.decoder.final_norm(dec_x)
        logits = self.decoder.lm_head(dec_x)
        return logits

class TranslationDataset(Dataset):
    def __init__(self, src_path, tgt_path, src_tokenizer, tgt_tokenizer, max_len=128):
        self.src_tokenizer = src_tokenizer
        self.tgt_tokenizer = tgt_tokenizer
        self.max_len = max_len
        print(f"Loading parallel data...")
        with open(src_path, 'r', encoding='utf-8') as f_src, open(tgt_path, 'r', encoding='utf-8') as f_tgt:
            self.src_lines = [line.strip() for line in f_src.readlines()]
            self.tgt_lines = [line.strip() for line in f_tgt.readlines()]

    def __len__(self): return len(self.src_lines)

    def __getitem__(self, idx):
        src_encoding = self.src_tokenizer(str(self.src_lines[idx]), truncation=True, max_length=self.max_len, padding='max_length', return_tensors="pt")
        tgt_encoding = self.tgt_tokenizer(str(self.tgt_lines[idx]), truncation=True, max_length=self.max_len, padding='max_length', return_tensors="pt")
        return {'src_ids': src_encoding['input_ids'].squeeze(0), 'src_mask': src_encoding['attention_mask'].squeeze(0), 'tgt_ids': tgt_encoding['input_ids'].squeeze(0)}

def train_translation():
    hi_tokenizer = AutoTokenizer.from_pretrained("l3cube-pune/hindi-bert-v2")
    mr_tokenizer = AutoTokenizer.from_pretrained("l3cube-pune/marathi-bert-v2")
    
    train_hi_path = "/kaggle/input/datasets/mradulnamdeo/iit-delhi-adivaani/train.hi"
    train_mr_path = "/kaggle/input/datasets/mradulnamdeo/iit-delhi-adivaani/train.mr"
    
    dataset = TranslationDataset(train_hi_path, train_mr_path, hi_tokenizer, mr_tokenizer, max_len=128)
    
    dataloader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=2)
    
    encoder = CustomBERTEncoder(vocab_size=hi_tokenizer.vocab_size).to(device)
    decoder = CustomGPTDecoder(vocab_size=mr_tokenizer.vocab_size).to(device)
    
    enc_ckpt = "/kaggle/input/datasets/akshy10/encoder-run/encoder_110M_epoch_4.pt"
    dec_ckpt = "/kaggle/input/datasets/akshy10/decpder-run/decoder_124M_epoch_4.pt"
    
    if os.path.exists(enc_ckpt) and os.path.exists(dec_ckpt):
        print("Loading pretrained weights...")
        encoder.load_state_dict(torch.load(enc_ckpt)['model_state_dict'])
        decoder.load_state_dict(torch.load(dec_ckpt)['model_state_dict'])
    else:
        print("\n[CRITICAL WARNING] Pretrained checkpoints not found! Check your paths.")
        return
    
    model = ConnectedTranslationModel(encoder, decoder).to(device)
    for param in model.encoder.parameters(): param.requires_grad = False
    for param in model.decoder.parameters(): param.requires_grad = False
    for param in model.cross_attention_layers.parameters(): param.requires_grad = True
    
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-4)
    criterion = nn.CrossEntropyLoss(ignore_index=mr_tokenizer.pad_token_id)
    scaler = torch.amp.GradScaler('cuda') 

    epochs = 4
    print(f"\nStarting Translation Fine-Tuning Phase ({epochs} Epochs)...")
    model.train()
    
    os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
    
    for epoch in range(epochs):
        if epoch == 1:
            print("Unfreezing base models for full fine-tuning...")
            for param in model.parameters(): param.requires_grad = True
            
            del optimizer
            torch.cuda.empty_cache()
            optimizer = optim.AdamW(model.parameters(), lr=5e-5) 
            
        total_loss = 0
        loop = tqdm(dataloader, leave=True)
        for batch in loop:
            src_ids, src_mask, tgt_ids = batch['src_ids'].to(device), batch['src_mask'].to(device), batch['tgt_ids'].to(device)
            tgt_input = tgt_ids
            
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                logits = model(src_ids, tgt_input, src_mask=src_mask)
                shift_logits = logits[:, :-1, :].contiguous()
                shift_labels = tgt_ids[:, 1:].contiguous()
                loss = criterion(shift_logits.view(-1, mr_tokenizer.vocab_size), shift_labels.view(-1))
                
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            
            total_loss += loss.item()
            loop.set_description(f"Epoch [{epoch+1}/{epochs}]")
            loop.set_postfix(loss=loss.item())
            
        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1} Completed | Average Loss: {avg_loss:.4f}")
        torch.save(model.state_dict(), f"/kaggle/working/checkpoints/final_translation_model_epoch_{epoch+1}.pt")
        
    print("\nPhase 3 Complete.")

if __name__ == "__main__":
    train_translation()

Writing train_translation.py


In [8]:
!python train_translation.py

In [9]:
%%writefile translate.py
import torch
from transformers import AutoTokenizer
import warnings
warnings.filterwarnings("ignore")

from architecture import RMSNorm, RotaryPositionalEmbedding, GroupedQueryAttention, FeedForward
from pretrain_encoder import CustomBERTEncoder
from pretrain_decoder import CustomGPTDecoder
from train_translation import ConnectedTranslationModel

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def load_model(epoch=3):
    print("Loading tokenizers and architecture...")
    hi_tokenizer = AutoTokenizer.from_pretrained("l3cube-pune/hindi-bert-v2")
    mr_tokenizer = AutoTokenizer.from_pretrained("l3cube-pune/marathi-bert-v2")

    encoder = CustomBERTEncoder(vocab_size=hi_tokenizer.vocab_size).to(device)
    decoder = CustomGPTDecoder(vocab_size=mr_tokenizer.vocab_size).to(device)
    model = ConnectedTranslationModel(encoder, decoder).to(device)

    # Pointing to your final 3rd epoch weights
    model_path = f"/kaggle/input/datasets/mradulnamdeo/translation-run-data/final_translation_model_epoch_3.pt"
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval() 
    return model, hi_tokenizer, mr_tokenizer

def translate(text, model, hi_tokenizer, mr_tokenizer, max_length=60):
    # 1. Encode the Hindi text
    src_encoding = hi_tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)
    src_ids = src_encoding['input_ids']
    src_mask = src_encoding['attention_mask']

    # 2. Start the Marathi generation with the [CLS] token
    tgt_ids = torch.tensor([[mr_tokenizer.cls_token_id]]).to(device)

    with torch.no_grad():
        # 3. The Autoregressive Loop
        for _ in range(max_length):
            # Pass the current state through the whole bridge
            logits = model(src_ids, tgt_ids, src_mask=src_mask)
            
            # Grab the single most likely next word
            next_token_id = logits[:, -1, :].argmax(dim=-1).unsqueeze(1)
            
            # Append it to the sentence
            tgt_ids = torch.cat([tgt_ids, next_token_id], dim=1)

            # Stop if the model predicts the End-Of-Sentence [SEP] token
            if next_token_id.item() in [mr_tokenizer.sep_token_id, mr_tokenizer.pad_token_id]:
                break

    # 4. Decode the IDs back into readable Marathi
    return mr_tokenizer.decode(tgt_ids[0], skip_special_tokens=True)

if __name__ == "__main__":
    model, hi_tok, mr_tok = load_model(epoch=3) 
    
    print("\n" + "="*40)
    print(" INFERENCE ENGINE READY ")
    print("="*40 + "\n")
    
    test_sentences = [
        "भारत एक महान देश है।",
        "मैं आज बहुत खुश हूँ।",
        "आप कैसे हैं?"
    ]
    
    for sentence in test_sentences:
        marathi_output = translate(sentence, model, hi_tok, mr_tok)
        print(f"Hindi Input : {sentence}")
        print(f"Marathi Out : {marathi_output}")
        print("-" * 40)

Writing translate.py


In [10]:
!python translate.py

Loading tokenizers and architecture...
tokenizer_config.json: 100%|███████████████████| 453/453 [00:00<00:00, 1.12MB/s]
vocab.txt: 3.16MB [00:00, 89.3MB/s]
tokenizer.json: 6.41MB [00:00, 108MB/s]
tokenizer_config.json: 100%|███████████████████| 455/455 [00:00<00:00, 2.08MB/s]
vocab.txt: 3.16MB [00:00, 96.0MB/s]
tokenizer.json: 6.41MB [00:00, 108MB/s]
special_tokens_map.json: 100%|██████████████████| 125/125 [00:00<00:00, 490kB/s]

 INFERENCE ENGINE READY 

Hindi Input : भारत एक महान देश है।
Marathi Out : भारत हा एक महान देश आहे.
----------------------------------------
Hindi Input : मैं आज बहुत खुश हूँ।
Marathi Out : आज मला अतिशय आनंद होतो आहे.
----------------------------------------
Hindi Input : आप कैसे हैं?
Marathi Out : तुम्ही कसे आहात?
----------------------------------------


In [12]:
%%writefile automated_evaluation_pipeline.py
import torch
import torch.nn as nn
from transformers import AutoTokenizer
import os
import sacrebleu
import matplotlib.pyplot as plt
from tqdm import tqdm

# Import our custom architecture and translation function
from pretrain_encoder import CustomBERTEncoder
from pretrain_decoder import CustomGPTDecoder
from train_translation import ConnectedTranslationModel
from translate import translate 

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def master_pipeline():
    print("Initializing Master Evaluation Pipeline...")
    hi_tok = AutoTokenizer.from_pretrained("l3cube-pune/hindi-bert-v2")
    mr_tok = AutoTokenizer.from_pretrained("l3cube-pune/marathi-bert-v2")

    train_hi_path = "/kaggle/input/datasets/mradulnamdeo/iit-delhi-adivaani/train.hi"
    train_mr_path = "/kaggle/input/datasets/mradulnamdeo/iit-delhi-adivaani/train.mr"
    val_hi_path   = "/kaggle/input/datasets/mradulnamdeo/iit-delhi-adivaani/test.hi"
    val_mr_path   = "/kaggle/input/datasets/mradulnamdeo/iit-delhi-adivaani/test.mr"

    # Load 100 sentences
    def load_fast_sample(hi_path, mr_path, n=100):
        with open(hi_path, 'r', encoding='utf-8') as f1, open(mr_path, 'r', encoding='utf-8') as f2:
            return [l.strip() for l in f1.readlines()][:n], [l.strip() for l in f2.readlines()][:n]

    print("Loading datasets...")
    t_hi, t_mr = load_fast_sample(train_hi_path, train_mr_path)
    v_hi, v_mr = load_fast_sample(val_hi_path, val_mr_path)

    encoder = CustomBERTEncoder(vocab_size=hi_tok.vocab_size).to(device)
    decoder = CustomGPTDecoder(vocab_size=mr_tok.vocab_size).to(device)
    model = ConnectedTranslationModel(encoder, decoder).to(device)
    criterion = nn.CrossEntropyLoss(ignore_index=mr_tok.pad_token_id)

    # Data Storage Arrays for the Plots
    epochs = [1, 2, 3]
    metrics = {
        'train_loss': [], 'val_loss': [],
        'train_bleu': [], 'val_bleu': [],
        'train_chrf': [], 'val_chrf': []
    }

    # Helper function for fast batched loss calculation
    def calculate_loss(hi_lines, mr_lines):
        total_loss = 0
        batch_size = 8
        batches = max(1, len(hi_lines) // batch_size)
        
        with torch.no_grad(): 
            for i in range(0, len(hi_lines), batch_size):
                b_hi, b_mr = hi_lines[i:i+batch_size], mr_lines[i:i+batch_size]
                src_enc = hi_tok(b_hi, return_tensors="pt", truncation=True, max_length=128, padding="max_length").to(device)
                tgt_enc = mr_tok(b_mr, return_tensors="pt", truncation=True, max_length=128, padding="max_length").to(device)
                
                logits = model(src_enc['input_ids'], tgt_enc['input_ids'], src_mask=src_enc['attention_mask'])
                shift_logits = logits[:, :-1, :].contiguous()
                shift_labels = tgt_enc['input_ids'][:, 1:].contiguous()
                
                loss = criterion(shift_logits.view(-1, mr_tok.vocab_size), shift_labels.view(-1))
                total_loss += loss.item()
                
        return total_loss / batches

    # Helper function for NLP metrics (Slow due to autoregressive generation)
    def calculate_nlp_metrics(src_lines, ref_lines, desc):
        predictions = []
        for sentence in tqdm(src_lines, desc=desc, leave=False):
            pred = translate(sentence, model, hi_tok, mr_tok)
            predictions.append(pred)
            
        bleu = sacrebleu.corpus_bleu(predictions, [ref_lines]).score
        chrf = sacrebleu.corpus_chrf(predictions, [ref_lines]).score
        return bleu, chrf

    print("\n" + "="*50)
    print(" COMMENCING EVALUATION AND PLOT GENERATION")
    print("="*50)

    for epoch in epochs:
        path = f"/kaggle/input/datasets/mradulnamdeo/translation-run-data/final_translation_model_epoch_{epoch}.pt"
        
        if not os.path.exists(path):
            print(f"\n[CRITICAL ERROR] Epoch {epoch} checkpoint missing at: {path}")
            continue
            
        print(f"\nEvaluating Epoch {epoch}...")
        model.load_state_dict(torch.load(path, map_location=device))
        model.eval()

        # 1. Calculate Loss (Fast)
        train_loss = calculate_loss(t_hi, t_mr)
        val_loss = calculate_loss(v_hi, v_mr)
        
        # 2. Calculate NLP Metrics (Slow)
        train_bleu, train_chrf = calculate_nlp_metrics(t_hi, t_mr, f"Epoch {epoch} Train Gen")
        val_bleu, val_chrf = calculate_nlp_metrics(v_hi, v_mr, f"Epoch {epoch} Val Gen")

        # 3. Store Results
        metrics['train_loss'].append(train_loss); metrics['val_loss'].append(val_loss)
        metrics['train_bleu'].append(train_bleu); metrics['val_bleu'].append(val_bleu)
        metrics['train_chrf'].append(train_chrf); metrics['val_chrf'].append(val_chrf)
        
        print(f"--> Epoch {epoch} Final Scores:")
        print(f"    Loss   | Train: {train_loss:.3f} | Val: {val_loss:.3f}")
        print(f"    BLEU   | Train: {train_bleu:.2f} | Val: {val_bleu:.2f}")
        print(f"    CHRF++ | Train: {train_chrf:.2f} | Val: {val_chrf:.2f}")

    # ==========================================
    # FINAL STEP: AUTOMATIC PLOTTING
    # ==========================================
    if len(metrics['train_loss']) > 0:
        print("\nDrawing and saving IIT Delhi Plots...")
        fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 6))
        fig.suptitle("IIT Delhi Translation Task: Full Quantitative Evaluation", fontsize=18, fontweight='bold', y=1.05)

        valid_epochs = epochs[:len(metrics['train_loss'])]

        # Plot 1: Loss
        ax1.plot(valid_epochs, metrics['train_loss'], label='Train Loss', marker='o', color='#1f77b4', linewidth=2.5)
        ax1.plot(valid_epochs, metrics['val_loss'], label='Validation Loss', marker='s', color='#ff7f0e', linewidth=2.5)
        ax1.set_title("Cross-Entropy Loss Curve", fontsize=14, fontweight='bold')
        ax1.set_xlabel("Epochs", fontsize=12); ax1.set_ylabel("Loss", fontsize=12)
        ax1.set_xticks(valid_epochs); ax1.grid(True, linestyle='--', alpha=0.7); ax1.legend()

        # Plot 2: BLEU
        ax2.plot(valid_epochs, metrics['train_bleu'], label='Train BLEU', marker='o', color='#2ca02c', linewidth=2.5)
        ax2.plot(valid_epochs, metrics['val_bleu'], label='Validation BLEU', marker='s', color='#d62728', linewidth=2.5)
        ax2.set_title("BLEU Score (0-100)", fontsize=14, fontweight='bold')
        ax2.set_xlabel("Epochs", fontsize=12); ax2.set_ylabel("BLEU", fontsize=12)
        ax2.set_xticks(valid_epochs)
        ax2.set_ylim([0, max(max(metrics['train_bleu'], default=0), max(metrics['val_bleu'], default=0)) + 10])
        ax2.grid(True, linestyle='--', alpha=0.7); ax2.legend()

        # Plot 3: CHRF++
        ax3.plot(valid_epochs, metrics['train_chrf'], label='Train CHRF++', marker='o', color='#9467bd', linewidth=2.5)
        ax3.plot(valid_epochs, metrics['val_chrf'], label='Validation CHRF++', marker='s', color='#8c564b', linewidth=2.5)
        ax3.set_title("CHRF++ Score (0-100)", fontsize=14, fontweight='bold')
        ax3.set_xlabel("Epochs", fontsize=12); ax3.set_ylabel("CHRF++", fontsize=12)
        ax3.set_xticks(valid_epochs)
        ax3.set_ylim([0, max(max(metrics['train_chrf'], default=0), max(metrics['val_chrf'], default=0)) + 10])
        ax3.grid(True, linestyle='--', alpha=0.7); ax3.legend()

        plt.tight_layout()
        save_path = "/kaggle/working/IIT_Delhi_Automated_Plots.png"
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"\nSUCCESS! Your final image is saved at: {save_path}")
        plt.show()

if __name__ == "__main__":
    master_pipeline()

Writing automated_evaluation_pipeline.py


In [13]:
!python automated_evaluation_pipeline.py

Initializing Master Evaluation Pipeline...
Loading datasets...

 COMMENCING EVALUATION AND PLOT GENERATION

Evaluating Epoch 1...
--> Epoch 1 Final Scores:                                                       
    Loss   | Train: 4.597 | Val: 4.490
    BLEU   | Train: 0.33 | Val: 0.35
    CHRF++ | Train: 11.31 | Val: 11.42

Evaluating Epoch 2...
--> Epoch 2 Final Scores:                                                       
    Loss   | Train: 3.031 | Val: 3.485
    BLEU   | Train: 5.22 | Val: 3.53
    CHRF++ | Train: 27.06 | Val: 25.20

Evaluating Epoch 3...
--> Epoch 3 Final Scores:                                                       
    Loss   | Train: 2.016 | Val: 2.807
    BLEU   | Train: 14.09 | Val: 7.69
    CHRF++ | Train: 40.61 | Val: 35.54

Drawing and saving IIT Delhi Plots...

SUCCESS! Your final image is saved at: /kaggle/working/IIT_Delhi_Automated_Plots.png
Figure(2000x600)
